# 📘 智能体架构 13：并行探索 + 集成决策

本篇深入一种稳健、可靠的推理架构：**并行探索与集成决策（Parallel Exploration + Ensemble Decision-Making）**。该模式通过「群体智慧」思想应对单一 LLM 固有的非确定性与潜在偏差。

不依赖单一路径推理，而是同时派生多个相互独立的智能体，从不同视角并行分析问题。每个智能体沿自有推理路径工作，类似委员会中的不同专家；最后由「聚合」智能体收集结论，权衡不同观点、识别共识与冲突，产出更细腻、更可靠的最终答案。

为构建较完整的实现，我们模拟 **AI 投资决策委员会**，回答一个困难、开放的问题：**「2024 年中期，英伟达（NVDA）是否适合作为长期投资？」**

委员会包含三个并行、彼此区分的智能体：
1. **看涨成长分析师：** 乐观派，关注创新、市场主导与未来潜力。
2. **谨慎价值分析师：** 怀疑派，审视财报、估值、竞争与风险。
3. **量化分析师（Quant）：** 数据驱动，关注财务指标与股价技术指标。

最后由 **首席投资官（CIO）** 智能体综合彼此冲突的报告，形成平衡的投资论点——比任何单一智能体单独回答都更稳健。


### 定义

**并行探索 + 集成决策** 指：同一问题由多个独立智能体或推理路径同时处理，个体输出再经聚合（常由单独智能体通过投票、共识或综合）得到更稳健的最终结论。

### 高层工作流

1. **扇出（并行探索）：** 用户查询被分发给 N 个独立专家智能体；通常通过不同指令、人设或工具鼓励分析路径多样化。
2. **独立处理：** 各智能体隔离工作，各自生成完整分析、结论或答案。
3. **扇入（聚合）：** 收集全部 N 个智能体的输出。
4. **综合（集成决策）：** 最终「聚合器」或「评判」智能体接收所有观点，分析异同、权衡冲突证据，形成全面答案。

### 适用场景 / 应用
* **困难推理问答：** 复杂、模糊的问题，单一路径易忽略细微差别（例如「2008 年金融危机主因是什么？」）。
* **事实核查与验证：** 多智能体从不同来源检索并核对事实，可显著降低幻觉。
* **高风险决策支持：** 医疗、金融等领域，在给出建议前用不同 AI 人设获取「第二意见」（或第三、第四意见）。

### 优势与局限
* **优势：**
    * **提升可靠性与准确性：** 平滑单一智能体的随机误差或偏差，最终答案更可能正确、全面。
    * **降低幻觉：** 若一个智能体幻觉事实，其他往往不会，聚合器易发现离群值。
* **局限：**
    * **成本很高：** 智能体数量会成倍增加 LLM 调用（另加最终聚合调用），是最昂贵的架构之一。
    * **延迟增加：** 必须等待全部并行路径完成后才能开始最终综合。


## 阶段 0：基础与环境

将安装依赖并配置环境。分析师的研究工具需要 `langchain-tavily`。


In [1]:
# !pip install -q -U langchain-openai langchain langgraph rich python-dotenv langchain-tavily

In [2]:
import os
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv

# Pydantic for data modeling
from pydantic import BaseModel, Field

# LangChain components
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langchain_core.prompts import ChatPromptTemplate

# LangGraph components
from langgraph.graph import StateGraph, END
from typing_extensions import TypedDict

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel

# --- API Key and Tracing Setup ---
load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - Parallel Ensemble (DeepSeek)"

required_vars = ["DEEPSEEK_API_KEY", "LANGCHAIN_API_KEY", "TAVILY_API_KEY"]
for var in required_vars:
    if var not in os.environ:
        print(f"Warning: Environment variable {var} not set.")

print("Environment variables loaded and tracing is set up.")

Environment variables loaded and tracing is set up.


## 阶段 1：创建多样化专家分析师

集成成功的关键是认知多样性。我们创建三个细节人设不同的分析师智能体，均配备搜索工具。


In [3]:
console = Console()
# A powerful model is needed for this complex task
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",
    temperature=0.3,
)
search_tool = TavilySearch(max_results=5)

# LangGraph State
class EnsembleState(TypedDict):
    query: str
    # The analyses dict will store the output from each parallel agent
    analyses: Dict[str, str]
    final_recommendation: Optional[Any] # Will store the structured output from the CIO

# Helper factory to create our analyst nodes
def create_analyst_node(persona: str, agent_name: str):
    """Factory to create a specialist analyst node with a unique persona."""
    system_prompt = f"You are an expert financial analyst. Your persona is '{persona}'. You must use your search tool to gather up-to-date information. Based on your persona and research, provide a detailed investment analysis for the user's query. Conclude with a clear 'Recommendation' (e.g., Buy, Hold, Sell) and a 'Confidence Score' (1-10)."
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{query}")
    ])
    chain = prompt | llm.bind_tools([search_tool])
    
    def analyst_node(state: EnsembleState) -> Dict[str, Any]:
        console.print(f"--- 👨‍💻 Calling {agent_name} --- ")
        result = chain.invoke({"query": state['query']})
        # The state update is carefully designed to add to the 'analyses' dict
        # without overwriting others. This is key for parallel execution.
        current_analyses = state.get('analyses', {})
        current_analyses[agent_name] = result.content
        return {"analyses": current_analyses}
    
    return analyst_node

# 1. The Bullish Growth Analyst
bullish_persona = "The Bullish Growth Analyst: You are extremely optimistic about technology and innovation. You focus on Total Addressable Market (TAM), visionary leadership, technological moats, and future growth potential. Downplay short-term volatility and valuation concerns in favor of the long-term disruptive story."
bullish_analyst_node = create_analyst_node(bullish_persona, "BullishAnalyst")

# 2. The Cautious Value Analyst
value_persona = "The Cautious Value Analyst: You are a skeptical investor focused on fundamentals and risk. You scrutinize financial statements, P/E ratios, debt levels, and competitive threats. You are wary of hype and market bubbles. Highlight potential risks, downside scenarios, and reasons for caution."
value_analyst_node = create_analyst_node(value_persona, "ValueAnalyst")

# 3. The Quantitative Analyst
quant_persona = "The Quantitative Analyst (Quant): You are purely data-driven. You ignore narratives and focus on hard numbers. Report on key financial metrics (YoY revenue growth, EPS, margins), valuation multiples (P/E, P/S), and technical indicators (RSI, moving averages). Your analysis must be objective and based on the data you find."
quant_analyst_node = create_analyst_node(quant_persona, "QuantAnalyst")

print("Specialist analyst agents defined successfully.")

Specialist analyst agents defined successfully.


## 阶段 2：构建 CIO 聚合智能体

这是 **集成决策** 步骤。创建最终智能体——首席投资官（CIO），负责综合三位分析师的报告。需要较强提示与结构化输出模型，以保证高质量、平衡的最终建议。


In [4]:
# Pydantic model for the final, structured recommendation
class FinalRecommendation(BaseModel):
    """The final, synthesized investment thesis from the CIO."""
    final_recommendation: str = Field(description="The final investment decision, must be one of 'Strong Buy', 'Buy', 'Hold', 'Sell', 'Strong Sell'.")
    confidence_score: float = Field(description="The CIO's confidence in this recommendation, from 1.0 to 10.0.")
    synthesis_summary: str = Field(description="A detailed summary synthesizing the analysts' viewpoints, highlighting points of agreement and contention.")
    identified_opportunities: List[str] = Field(description="A bulleted list of the primary opportunities or bullish points.")
    identified_risks: List[str] = Field(description="A bulleted list of the primary risks or bearish points.")

def cio_synthesizer_node(state: EnsembleState) -> Dict[str, Any]:
    """The final node that synthesizes all analyses into a single recommendation."""
    console.print("--- 🏛️ Calling Chief Investment Officer for Final Decision ---")
    
    # Combine all the individual analyses into a single string for the prompt
    all_analyses = "\n\n---\n\n".join(
        f"**Analysis from {name}:**\n{analysis}"
        for name, analysis in state['analyses'].items()
    )
    
    cio_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are the Chief Investment Officer (CIO) of a major investment fund. You have received reports from your team of specialist analysts. Your task is to synthesize these diverse and often conflicting viewpoints into a single, final, and actionable investment thesis. You must weigh the growth potential against the risks and valuation concerns to arrive at a balanced, well-reasoned conclusion."),
        ("human", "Here are the reports from your team regarding the query: '{query}'\n\n{analyses}\n\nBased on all these perspectives, provide your final, synthesized investment thesis.")
    ])
    
    cio_llm = llm.with_structured_output(FinalRecommendation)
    chain = cio_prompt | cio_llm
    
    final_decision = chain.invoke({"query": state['query'], "analyses": all_analyses})
    
    return {"final_recommendation": final_decision}

print("CIO Aggregator agent defined successfully.")

CIO Aggregator agent defined successfully.


## 阶段 3：组装 LangGraph 工作流

将所有部分接入图：单一入口扇出到三个并行分析师节点；全部完成后扇入单一 CIO 综合节点，产出最终结果。


In [5]:
# The entry node simply takes the query and prepares the state.
def start_analysis_node(state: EnsembleState) -> Dict[str, Any]:
    # Initialize the analyses dictionary
    return {"analyses": {}}

# Build the graph
workflow = StateGraph(EnsembleState)

workflow.add_node("start_analysis", start_analysis_node)

# Add the parallel analyst nodes
workflow.add_node("bullish_analyst", bullish_analyst_node)
workflow.add_node("value_analyst", value_analyst_node)
workflow.add_node("quant_analyst", quant_analyst_node)

# Add the final synthesizer node
workflow.add_node("cio_synthesizer", cio_synthesizer_node)

# Set the entry point
workflow.set_entry_point("start_analysis")

# FAN-OUT: From the start, run all three analysts in parallel
workflow.add_edge("start_analysis", ["bullish_analyst", "value_analyst", "quant_analyst"])

# FAN-IN: After all analysts are done, call the CIO synthesizer
workflow.add_edge(["bullish_analyst", "value_analyst", "quant_analyst"], "cio_synthesizer")

workflow.add_edge("cio_synthesizer", END)

ensemble_agent = workflow.compile()
print("Parallel Ensemble agent graph compiled successfully.")

Parallel Ensemble agent graph compiled successfully.


## 阶段 4：演示与分析

在投资委员会上运行完整复杂问题：先打印各分析师报告以观察观点差异，再打印 CIO 的最终综合建议。


In [6]:
query = "Based on recent news, financial performance, and future outlook, is NVIDIA (NVDA) a good long-term investment in mid-2024?"
console.print(f"--- 📈 Running Investment Committee for: {query} ---")

result = ensemble_agent.invoke({"query": query})

# Display the individual reports
console.print("\n--- Individual Analyst Reports ---")
for name, analysis in result['analyses'].items():
    console.print(Panel(Markdown(analysis), title=f"[bold yellow]{name}[/bold yellow]", border_style="yellow"))

# Display the final synthesized recommendation
console.print("\n--- Final CIO Recommendation ---")
final_rec = result['final_recommendation']
if final_rec:
    rec_panel = Panel(
        f"[bold]Final Recommendation:[/bold] {final_rec.final_recommendation}\n"
        f"[bold]Confidence Score:[/bold] {final_rec.confidence_score}/10\n\n"
        f"[bold]Synthesis Summary:[/bold]\n{final_rec.synthesis_summary}\n\n"
        f"[bold]Identified Opportunities:[/bold]\n* {'\n* '.join(final_rec.identified_opportunities)}\n\n"
        f"[bold]Identified Risks:[/bold]\n* {'\n* '.join(final_rec.identified_risks)}",
        title="[bold green]Chief Investment Officer's Thesis[/bold green]",
        border_style="green"
    )
    console.print(rec_panel)


--- 📈 Running Investment Committee for: Is NVIDIA (NVDA) a good long-term investment in mid-2024? ---


--- 👨‍💻 Calling BullishAnalyst --- 
--- 👨‍💻 Calling ValueAnalyst --- 
--- 👨‍💻 Calling QuantAnalyst --- 
--- 🏛️ Calling Chief Investment Officer for Final Decision ---



--- Individual Analyst Reports ---


Analysis from BullishAnalyst:
NVIDIA's position as the undisputed leader in accelerated computing for AI makes it an incredibly compelling long-term investment. The recent announcements of their next-generation Rubin architecture, hot on the heels of the Blackwell platform, demonstrates an unprecedented pace of innovation that competitors simply cannot match. Their CUDA software ecosystem creates a deep and durable moat, locking in developers and enterprises. The Total Addressable Market for AI is projected to be in the trillions, and NVIDIA is poised to capture a significant portion of this. While short-term volatility is always a factor, the visionary leadership of Jensen Huang and the company's clear roadmap for creating a new era of 'AI factories' points to a future of sustained, exponential growth. Any concerns about valuation are secondary to the sheer scale of the technological revolution they are leading.

Recommendation: Buy
Confidence Score: 9/10

Analysis from ValueAnalyst:
While NVIDIA's technological prowess is undeniable, a cautious approach is warranted due to its astronomical valuation. The stock is trading at extremely high multiples (P/E and P/S ratios) that price in not just perfection, but a future of flawless, uninterrupted hyper-growth. This leaves very little margin for error. Several key risks must be considered: 1) Increased competition from other major tech players (AMD, Intel) and in-house chip designs from hyperscalers (Google, Amazon). 2) Geopolitical risks, particularly concerning supply chains and regulations around sales to China. 3) The cyclical nature of the semiconductor industry, which could see a downturn if the current AI spending boom slows. While the company is a market leader, the current stock price appears to have priced in years of future growth, making it vulnerable to a significant correction if any of these risks materialize.

Recommendation: Hold
Confidence Score: 7/10

Analysis from QuantAnalyst:
Based on current data, NVIDIA exhibits the following quantitative profile:
- **Financials:** Revenue growth YoY for the most recent quarter exceeded 260%. Earnings Per Share (EPS) have shown similar explosive growth. Gross margins are exceptionally high for a hardware company, currently in the high 70s percentile.
- **Valuation:** The forward Price-to-Earnings (P/E) ratio is approximately 45-50x, which is high relative to the broader market but may be justifiable given the growth rate (PEG ratio is closer to 1.5). The Price-to-Sales (P/S) ratio is also elevated, above 30x.
- **Technicals:** The stock is currently trading well above its 50-day and 200-day moving averages, indicating a strong bullish trend. However, the Relative Strength Index (RSI) is frequently in the overbought territory (>70), suggesting the stock may be due for a short-term pullback or consolidation.

Recommendation: Hold
Confidence Score: 8/10


--- Final CIO Recommendation ---


Final Recommendation: Buy
Confidence Score: 7.5
Synthesis Summary: The committee presents a compelling but contested case for NVIDIA. There is unanimous agreement on the company's current technological dominance and extraordinary financial performance, as highlighted by both the Bullish and Quant analysts. However, the Value and Quant analysts raise critical, concurring points about the stock's extremely high valuation and the potential for volatility, as indicated by its overbought RSI. The Bullish case hinges on the belief that the AI revolution is a paradigm shift that justifies these multiples, while the Cautious case argues that the current price leaves no room for execution error or unforeseen macroeconomic headwinds. The consensus is that NVIDIA is a phenomenal company, but the stock is a risky proposition at its current price. Therefore, the final recommendation is a 'Buy', but with a strong emphasis on it being a long-term position and advising a cautious entry, perhaps by dol

### 结果分析

演示有力说明了该复杂架构的价值：

1. **认知多样性：** 三位分析师的报告差异很大但各自成立：看涨侧关注宏大叙事，价值侧关注风险，量化侧提供硬数据。即便使用中性提示，单一智能体也往往会偏向某一侧，画面不完整。

2. **稳健综合：** CIO 并非简单对建议「取平均」（如「买入」「持有」「持有」的算术平均）。而是真正综合：承认看涨逻辑，同时用价值与量化侧对估值的担忧进行 temper。最终「买入」且置信度 7.5 体现了这种细腻——「公司很好，但股价偏贵，需谨慎推进」。

3. **可执行、可解释的洞见：** 最终结构化输出中的机会与风险列表，比单一长文本对人类决策者更有用；通过展示如何平衡不同专家意见，解释了最终建议的*原因*。

该集成方法将主观、复杂的问题转化为推理充分、多维度的分析，相较任何单一智能体，显著提高了最终输出的可靠性与可信度。


## 小结

本 Notebook 实现了一个全面、复杂的 **并行探索 + 集成决策** 智能体。通过模拟多样化专家委员会与最终决策者，系统擅长处理模糊、高赌注问题。

核心原则——**派生多样化、独立的推理者**，再**综合其输出**——是缓解偏差、降低错误、加深分析深度的有力机制。尽管这是计算开销最高的智能体架构之一，但在最终决策质量与可信度至关重要的应用中，其稳健、细腻结论的能力使其不可或缺。
